In [1336]:
import pandas as pd
import joblib
import numpy as np
import itertools

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import roc_auc_score, brier_score_loss, classification_report, f1_score, recall_score, precision_score, confusion_matrix, classification_report
from xgboost import XGBClassifier

In [1337]:
df = pd.read_csv("../data/framingham.csv")

In [1338]:
features = ["age","male","currentSmoker","cigsPerDay","diabetes","BMI","sysBP","totChol","glucose"]

In [1339]:
X = df[features]
y = df["TenYearCHD"]

In [1340]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=39, stratify=y)

In [1341]:
numeric_features = ["age","cigsPerDay","BMI","sysBP","totChol","glucose"]

In [1342]:
binary_features = ["male","currentSmoker","diabetes"]

In [1343]:
numeric_pipeline = Pipeline(steps=[("imputer", SimpleImputer(strategy="median", add_indicator=True)), ("scaler", StandardScaler())])

In [1344]:
binary_pipeline = Pipeline(steps=[("imputer", SimpleImputer(strategy="most_frequent"))])

In [1345]:
preprocessor = ColumnTransformer([("num", numeric_pipeline, numeric_features), ("bin", binary_pipeline, binary_features)])

In [1346]:
logreg = Pipeline(steps=[("preprocess", preprocessor), ("classifier", LogisticRegression(max_iter=2000, class_weight="balanced", solver="lbfgs"))])

In [1347]:
logreg.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocess', ...), ('classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('bin', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers

In [1348]:
from sklearn.model_selection import StratifiedKFold, cross_val_score

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=39)

auc_scores = cross_val_score(logreg, X_train, y_train, cv=cv, scoring="roc_auc", n_jobs=-1)

print("Cross Validated AUC scores:", auc_scores)
print("Mean CV AUC:", round(auc_scores.mean(), 4))
print("Std CV AUC:", round(auc_scores.std(), 4))

Cross Validated AUC scores: [0.70352953 0.74110032 0.71355002 0.71496834 0.74853525]
Mean CV AUC: 0.7243
Std CV AUC: 0.0173


In [1349]:
probs = logreg.predict_proba(X_test)[:,1]
preds = logreg.predict(X_test)

print("AUC Score:", roc_auc_score(y_test, probs))
print("Brier Score:", brier_score_loss(y_test, probs))

print("Classification Report:")
print(classification_report(y_test, preds))

AUC Score: 0.7307091028668153
Brier Score: 0.20426227568135244
Classification Report:
              precision    recall  f1-score   support

           0       0.92      0.69      0.79       719
           1       0.28      0.67      0.40       129

    accuracy                           0.69       848
   macro avg       0.60      0.68      0.59       848
weighted avg       0.82      0.69      0.73       848



In [1350]:
calibrated_logreg = CalibratedClassifierCV(logreg, method="isotonic",cv=5)

calibrated_logreg.fit(X_train, y_train)

cal_probs = calibrated_logreg.predict_proba(X_test)[:,1]
cal_preds = calibrated_logreg.predict(X_test)

print("Calibrated AUC Score:", roc_auc_score(y_test, cal_probs))
print("Calibrated Brier Score:", brier_score_loss(y_test, cal_probs))

print("Confusion Matrix:")
print(confusion_matrix(y_test, cal_preds))
print("Classification Report:")
print(classification_report(y_test, cal_preds))


Calibrated AUC Score: 0.7309409062975062
Calibrated Brier Score: 0.11697513624108559
Confusion Matrix:
[[713   6]
 [121   8]]
Classification Report:
              precision    recall  f1-score   support

           0       0.85      0.99      0.92       719
           1       0.57      0.06      0.11       129

    accuracy                           0.85       848
   macro avg       0.71      0.53      0.52       848
weighted avg       0.81      0.85      0.80       848



In [1351]:
thresholds = np.linspace(0.05, 0.4, 200)

rows = []
for t in thresholds:
    preds = (cal_probs >= t).astype(int)
    rows.append({
        "threshold": t,
        "precision": precision_score(y_test, preds, zero_division=0),
        "recall": recall_score(y_test, preds, zero_division=0),
        "f1": f1_score(y_test, preds, zero_division=0),
        "positive_rate": preds.mean()
    })


best_f1 = max(rows, key=lambda r: r["f1"])

min_precision = 0.25  
candidates = [r for r in rows if r["precision"] >= min_precision]
best_screen = max(candidates, key=lambda r: r["recall"]) if candidates else None

print("Best F1 Threshold")
print(f"Threshold:{best_f1['threshold']}")
print(f"Precision:{best_f1['precision']}")
print(f"Recall:{best_f1['recall']}")
print(f"F1-score:{best_f1['f1']}")
print(f"Positive rate:{best_f1['positive_rate']}")

if best_screen is not None:
    print("\nBest Screening Threshold")
    print(f"Threshold:{best_screen['threshold']}")
    print(f"Precision:{best_screen['precision']}")
    print(f"Recall:{best_screen['recall']}")
    print(f"F1-score:{best_screen['f1']}")
    print(f"Positive rate:{best_screen['positive_rate']}")
else:
    print("No screening threshold met the minimum precision requirement.")


Best F1 Threshold
Threshold:0.1748743718592965
Precision:0.296551724137931
Recall:0.6666666666666666
F1-score:0.4105011933174224
Positive rate:0.3419811320754717

Best Screening Threshold
Threshold:0.13090452261306534
Precision:0.25340599455040874
Recall:0.7209302325581395
F1-score:0.375
Positive rate:0.43278301886792453


In [1352]:
best_t = float(best_f1["threshold"])
print("Threshold:", round(best_t, 3))

Threshold: 0.175


In [1353]:
final_preds = (cal_probs >= best_t).astype(int)

print("Confusion Matrix:")
print(confusion_matrix(y_test, final_preds))
print("Classification Report:")
print(classification_report(y_test, final_preds, zero_division=0))

Confusion Matrix:
[[515 204]
 [ 43  86]]
Classification Report:
              precision    recall  f1-score   support

           0       0.92      0.72      0.81       719
           1       0.30      0.67      0.41       129

    accuracy                           0.71       848
   macro avg       0.61      0.69      0.61       848
weighted avg       0.83      0.71      0.75       848



In [1354]:
def get_percentiles(series, percentiles=(0.10, 0.50, 0.90)):
    series = series.dropna()

    if len(series) == 0:
        return []

    values = [series.quantile(p) for p in percentiles]

    return [int(round(v)) for v in values]


plausible_values = {}

for col in ["sysBP", "totChol", "glucose"]:
    plausible_values[col] = get_percentiles(X_train[col], percentiles=(0.10, 0.50, 0.90))

smokers_only = X_train[X_train["currentSmoker"] == 1]
plausible_values["cigsPerDay_if_smoker"] = get_percentiles(smokers_only["cigsPerDay"], percentiles=(0.10, 0.50, 0.90))

print("Plausible values (low / typical / high) learned from training data:")
for k, v in plausible_values.items():
    print(f"{k}: {v}")

Plausible values (low / typical / high) learned from training data:
sysBP: [108, 128, 162]
totChol: [184, 234, 292]
glucose: [65, 78, 98]
cigsPerDay_if_smoker: [5, 20, 30]


In [1355]:
final_logreg = {"model": calibrated_logreg, "threshold":float(best_t), "features":features, "plausible_values":plausible_values,}

In [1356]:
joblib.dump(final_logreg, "../models/heartsmart_model.pkl")

['../models/heartsmart_model.pkl']

In [1357]:
pos_weight = (y_train==0).sum() / (y_train==1).sum()

xgb_model = XGBClassifier(
    n_estimators=500,
    max_depth=3,
    learning_rate=0.03,
    subsample=0.9,
    colsample_bytree=0.9,
    min_child_weight=5,
    gamma=0.1,
    scale_pos_weight=pos_weight,
    eval_metric="logloss",
    random_state=39
)

In [1358]:
xgb_model.fit(X_train, y_train)

,"objective objective: typing.Union[str, xgboost.sklearn._SklObjWProto, typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]], NoneType]Specify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'binary:logistic'
,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,0.9
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes f

In [1359]:
xgb_probs = xgb_model.predict_proba(X_test)[:, 1]

xgb_preds = (xgb_probs >= 0.5).astype(int)

print("XGBoost Performance")
print("AUC Score:", round(roc_auc_score(y_test, xgb_probs), 4))
print("Brier Score :", round(brier_score_loss(y_test, xgb_probs), 4))

print("Classification Report:")
print(classification_report(y_test, xgb_preds, zero_division=0))

XGBoost Performance
AUC Score: 0.7012
Brier Score : 0.1933
Classification Report:
              precision    recall  f1-score   support

           0       0.90      0.71      0.79       719
           1       0.25      0.54      0.34       129

    accuracy                           0.68       848
   macro avg       0.57      0.63      0.57       848
weighted avg       0.80      0.68      0.72       848



In [1360]:
from sklearn.calibration import CalibratedClassifierCV

xgb_cal = CalibratedClassifierCV(xgb_model, method="isotonic", cv=5)

xgb_cal.fit(X_train, y_train)

xgb_cal_probs = xgb_cal.predict_proba(X_test)[:, 1]

print("Calibrated XGBoost")
print("Calibrated AUC Score:", round(roc_auc_score(y_test, xgb_cal_probs), 4))
print("Calibrated Brier Score:", round(brier_score_loss(y_test, xgb_cal_probs), 4))

Calibrated XGBoost
Calibrated AUC Score: 0.7025
Calibrated Brier Score: 0.1194
